**Data Cleaning and Preprocessing**

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

url = (
    "https://raw.githubusercontent.com/"
    "IBM/telco-customer-churn-on-icp4d/"
    "master/data/Telco-Customer-Churn.csv"
)

df = pd.read_csv(url)

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [ ]:
# Remove leading/trailing spaces and convert to numeric
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"].astype(str).str.strip(),
    errors="coerce"
)

print("Missing values after conversion:")
display(df.isnull().sum()[df.isnull().sum() > 0])

print("Duplicate rows:", df.duplicated().sum())

Missing values after conversion:


,0
TotalCharges,11


Duplicate rows: 0


In [ ]:
display(
    df.loc[
        df["TotalCharges"].isna(),
        ["tenure", "MonthlyCharges", "TotalCharges", "Churn"]
    ]
)

,tenure,MonthlyCharges,TotalCharges,Churn
488,0,52.55,NaN,No
753,0,20.25,NaN,No
936,0,80.85,NaN,No
1082,0,25.75,NaN,No
1340,0,56.05,NaN,No
3331,0,19.85,NaN,No
3826,0,25.35,NaN,No
4380,0,20.00,NaN,No
5218,0,19.70,NaN,No
6670,0,73.35,NaN,No


In [ ]:
# Remove exact duplicate records, if any
df = df.drop_duplicates().copy()

# Remove identifier
df = df.drop(columns=["customerID"])

# Separate target and input features
X = df.drop(columns=["Churn"])
y = df["Churn"].map({"No": 0, "Yes": 1})

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

print("\nTarget distribution:")
display(y.value_counts())

Feature matrix shape: (7043, 19)
Target shape: (7043,)

Target distribution:


,count
Churn,
0,5174
1,1869


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

print("\nTraining churn rate:")
print(round(y_train.mean() * 100, 2), "%")

print("Testing churn rate:")
print(round(y_test.mean() * 100, 2), "%")

Training samples: 5634
Testing samples: 1409

Training churn rate:
26.54 %
Testing churn rate:
26.54 %


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Identify column types from training data
numeric_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("Numerical features:", numeric_features)
print("\nCategorical features:", categorical_features)

# Numerical preprocessing
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine both
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

# Learn preprocessing only from training data
X_train_processed = preprocessor.fit_transform(X_train)

# Apply the learned transformations to test data
X_test_processed = preprocessor.transform(X_test)

print("\nProcessed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)

Numerical features: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']

Categorical features: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']

Processed training shape: (5634, 45)
Processed testing shape: (1409, 45)


In [ ]:
print("Training target distribution:")
display(y_train.value_counts(normalize=True).round(3))

print("\nTesting target distribution:")
display(y_test.value_counts(normalize=True).round(3))

print("\nMissing values in original feature matrix:")
display(X.isnull().sum()[X.isnull().sum() > 0])

print("\nProcessed matrix dimensions:")
print("Train:", X_train_processed.shape)
print("Test:", X_test_processed.shape)

Training target distribution:


,proportion
Churn,
0,0.735
1,0.265



Testing target distribution:


,proportion
Churn,
0,0.735
1,0.265



Missing values in original feature matrix:


,0
TotalCharges,11



Processed matrix dimensions:
Train: (5634, 45)
Test: (1409, 45)


In [ ]:
import joblib
import os
import zipfile

# Create a temporary project folder in Colab
PROJECT_DIR = "/content/Customer_Churn_Project"
os.makedirs(PROJECT_DIR, exist_ok=True)

# Save the cleaned training data
train_df = X_train.copy()
train_df["Churn"] = y_train

# Save the cleaned test data
test_df = X_test.copy()
test_df["Churn"] = y_test

train_df.to_csv(
    f"{PROJECT_DIR}/train_data.csv",
    index=False
)

test_df.to_csv(
    f"{PROJECT_DIR}/test_data.csv",
    index=False
)

# Save the fitted preprocessor as an optional artifact
joblib.dump(
    preprocessor,
    f"{PROJECT_DIR}/preprocessor.joblib"
)

print("Files created:")
print(os.listdir(PROJECT_DIR))

Files created:
['test_data.csv', 'train_data.csv', 'preprocessor.joblib']


In [ ]:
import shutil
from google.colab import files

zip_path = shutil.make_archive(
    "/content/Customer_Churn_Project",
    "zip",
    PROJECT_DIR
)

files.download(zip_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>